# Kitchen Receipts: train the document reader

Trains the AI model the app uses on the phone (Qwen3-VL 2B) on **exactly the questions the app asks it**, in Italian and
English: reading a product line, one number, the supplier's box, the totals, the header and the column headings; asking a
line again with the reason the first answer did not add up; and choosing which question to ask next (the agent's step).

Everything it learns from is **invented** (no real document): the app's own generator makes supplier documents as phone
photos, and the app's own code asks the questions, so training matches the phone exactly.

**Before you start:** Runtime ▸ Change runtime type ▸ **T4 GPU**. Add your Hugging Face token as a Colab secret named
`HF_TOKEN` (key icon on the left; the token needs *write* access). Then Runtime ▸ Run all. Nothing else to type.

The model is uploaded only if it does better than today's model on documents it never saw (the comparison is printed).

In [ ]:
# ---- Settings (nothing to change: your Hugging Face name is read from the HF_TOKEN secret)
from google.colab import userdata
from huggingface_hub import HfApi
HF_USER = HfApi(token=userdata.get("HF_TOKEN")).whoami()["name"]
print("Hugging Face account:", HF_USER)
REPO_NAME = "kitchen-reader-2b"    # the app recognises a model whose file name contains "kitchen"
N_TRAIN_DOCS = 1200                # about 15 questions each; ~4 hours on a free T4. Fewer if Colab stops early.
N_EVAL_DOCS = 80
N_EVAL_SAMPLES = 300               # questions used for the before/after comparison
EPOCHS = 1
BASE = "unsloth/Qwen3-VL-2B-Instruct"
REPO_URL = "https://github.com/Vyoki/Receipt-Logger"


In [ ]:
# ---- Tools: the app's code, Java (for the app's reading code), fonts for handwriting
!git clone -q --depth 1 $REPO_URL /content/app
!apt-get -qq install -y openjdk-17-jdk-headless > /dev/null
!pip -q install opencv-python-headless pillow
import os, urllib.request
os.makedirs("/content/fonts", exist_ok=True)
for path in ["caveat/Caveat%5Bwght%5D.ttf", "indieflower/IndieFlower-Regular.ttf", "kalam/Kalam-Regular.ttf",
             "homemadeapple/HomemadeApple-Regular.ttf", "reeniebeanie/ReenieBeanie-Regular.ttf", "shadowsintolight/ShadowsIntoLight.ttf",
             "nothingyoucoulddo/NothingYouCouldDo.ttf", "patrickhand/PatrickHand-Regular.ttf"]:
    name = path.split("/")[-1].replace("%5Bwght%5D", "")
    try:
        urllib.request.urlretrieve("https://raw.githubusercontent.com/google/fonts/main/ofl/" + path, "/content/fonts/" + name)
    except Exception as e:
        print("font not downloaded:", name, e)
print(os.listdir("/content/fonts"))

In [ ]:
# ---- Invented documents (photos + what the phone's OCR returns + the truth), then the app's questions on them
%cd /content/app
!python3 tools/training/gen_docs.py /content/docs_train --count $N_TRAIN_DOCS --seed 11 --fonts /content/fonts
!python3 tools/training/gen_docs.py /content/docs_eval --count $N_EVAL_DOCS --seed 97 --fonts /content/fonts
!bash tools/training/export.sh /content/docs_train /content/train_samples.jsonl 11
!bash tools/training/export.sh /content/docs_eval /content/eval_samples.jsonl 97
!python3 tools/training/build_dataset.py /content/train_samples.jsonl /content/docs_train /content/ds_train --eval-share 0
!python3 tools/training/build_dataset.py /content/eval_samples.jsonl /content/docs_eval /content/ds_eval --eval-share 0

In [ ]:
# ---- Training libraries
!pip -q install --upgrade unsloth unsloth_zoo
!pip -q install "trl>=0.20" "transformers>=4.57" peft accelerate bitsandbytes

In [ ]:
# ---- The data, loaded lazily (pictures read when needed: Colab's memory stays free)
import json, random, torch
from PIL import Image

class Samples(torch.utils.data.Dataset):
    def __init__(self, folder, limit=None, seed=0):
        self.folder = folder
        rows = [json.loads(l) for l in open(f"{folder}/train.jsonl")]
        random.Random(seed).shuffle(rows)
        self.rows = rows[:limit] if limit else rows
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, i):
        r = self.rows[i]
        img = Image.open(f"{self.folder}/{r['image']}").convert("RGB")
        user = [{"type": "image", "image": img}, {"type": "text", "text": r["messages"][0]["content"][1]["text"]}]
        return {"messages": [{"role": "user", "content": user}, r["messages"][1]]}

train = Samples("/content/ds_train", seed=1)
evals = Samples("/content/ds_eval", limit=N_EVAL_SAMPLES, seed=2)
print(len(train), "training questions;", len(evals), "for the comparison")

In [ ]:
# ---- The model, with small trainable adapters on both its eyes (vision) and its language
from unsloth import FastVisionModel
model, tokenizer = FastVisionModel.from_pretrained(BASE, load_in_4bit=False, use_gradient_checkpointing="unsloth")
model = FastVisionModel.get_peft_model(
    model, finetune_vision_layers=True, finetune_language_layers=True, finetune_attention_modules=True,
    finetune_mlp_modules=True, r=16, lora_alpha=16, lora_dropout=0, bias="none", random_state=3407,
)

In [ ]:
# ---- How well it answers: exact answers, and for JSON answers field by field
import re, collections

def ask(model, sample, max_new_tokens=160):
    msgs = [sample["messages"][0]]
    img = msgs[0]["content"][0]["image"]
    text = tokenizer.apply_chat_template([{"role": "user", "content": [{"type": "image"}, msgs[0]["content"][1]]}], add_generation_prompt=True)
    inputs = tokenizer(img, text, add_special_tokens=False, return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, use_cache=True, do_sample=False)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def fields(s):
    try:
        v = json.loads(s)
        return v if isinstance(v, dict) else None
    except Exception:
        return None

def score(model, data, n=None):
    FastVisionModel.for_inference(model)
    by = collections.defaultdict(lambda: [0, 0, 0, 0])   # exact, total, fields right, fields
    for i in range(n or len(data)):
        s = data[i]
        want = s["messages"][1]["content"][0]["text"]
        task = s["messages"][0]["content"][1]["text"].strip().split("\n")[0].split(" ")[0]
        got = ask(model, s)
        b = by[task]
        b[0] += got == want; b[1] += 1
        w, g = fields(want), fields(got)
        if w:
            b[3] += len(w); b[2] += sum(1 for k in w if g and g.get(k) == w[k])
    allx = sum(b[0] for b in by.values()) / max(1, sum(b[1] for b in by.values()))
    for t, b in sorted(by.items()):
        extra = f", fields {b[2] / b[3]:.0%}" if b[3] else ""
        print(f"{t:10s} exact {b[0] / b[1]:.0%} of {b[1]}{extra}")
    print(f"ALL       exact {allx:.1%}")
    return allx

print("Today's model (before training):")
before = score(model, evals)

In [ ]:
# ---- Training
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig
FastVisionModel.for_training(model)
trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, data_collator=UnslothVisionDataCollator(model, tokenizer), train_dataset=train,
    args=SFTConfig(
        per_device_train_batch_size=2, gradient_accumulation_steps=4, warmup_steps=30, num_train_epochs=EPOCHS,
        learning_rate=2e-4, logging_steps=20, optim="adamw_8bit", weight_decay=0.01, lr_scheduler_type="cosine",
        seed=3407, output_dir="/content/outputs", report_to="none", save_steps=500, save_total_limit=2,
        remove_unused_columns=False, dataset_text_field="", dataset_kwargs={"skip_prepare_dataset": True}, max_length=2048,
        fp16=not torch.cuda.is_bf16_supported(), bf16=torch.cuda.is_bf16_supported(),
    ),
)
trainer.train()

In [ ]:
# ---- After training: the same questions
print("Trained model:")
after = score(model, evals)
print(f"\nBefore {before:.1%}  ->  after {after:.1%}")
GOOD = after > before + 0.05
print("Better: it will be uploaded." if GOOD else "Not clearly better: it will NOT be uploaded.")

In [ ]:
# ---- Into the phone's format (GGUF): the model, compressed (Q4_K_M), and its vision part (Q8_0)
if GOOD:
    model.save_pretrained_merged("/content/merged", tokenizer, save_method="merged_16bit")
    !git clone -q --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
    !pip -q install -r /content/llama.cpp/requirements/requirements-convert_hf_to_gguf.txt
    !cmake -S /content/llama.cpp -B /content/llama.cpp/build -DLLAMA_CURL=OFF > /dev/null && cmake --build /content/llama.cpp/build --target llama-quantize -j 4 > /dev/null
    !python3 /content/llama.cpp/convert_hf_to_gguf.py /content/merged --outtype f16 --outfile /content/{REPO_NAME}-f16.gguf
    !python3 /content/llama.cpp/convert_hf_to_gguf.py /content/merged --mmproj --outtype q8_0 --outfile /content/mmproj-{REPO_NAME}-Q8_0.gguf
    !/content/llama.cpp/build/bin/llama-quantize /content/{REPO_NAME}-f16.gguf /content/{REPO_NAME}-Q4_K_M.gguf Q4_K_M
    !ls -la /content/*.gguf

In [ ]:
# ---- Upload to your Hugging Face account
if GOOD:
    from google.colab import userdata
    from huggingface_hub import HfApi
    api = HfApi(token=userdata.get("HF_TOKEN"))
    repo = f"{HF_USER}/{REPO_NAME}"
    api.create_repo(repo, exist_ok=True)
    card = f'''---
license: apache-2.0
base_model: Qwen/Qwen3-VL-2B-Instruct
---
# {REPO_NAME}

Qwen3-VL 2B trained to read food & beverage supplier documents (invoices, delivery notes, receipts, handwritten notes;
Italian and English) for the Kitchen Receipts app, on invented documents only. Before/after on held-out invented
documents: exact answers {before:.1%} -> {after:.1%}.
'''
    api.upload_file(path_or_fileobj=card.encode(), path_in_repo="README.md", repo_id=repo)
    for f in [f"{REPO_NAME}-Q4_K_M.gguf", f"mmproj-{REPO_NAME}-Q8_0.gguf"]:
        api.upload_file(path_or_fileobj=f"/content/{f}", path_in_repo=f, repo_id=repo)
    print(f"Uploaded: https://huggingface.co/{repo}")